# Expert Experiences Analysis

Validates the `.pkl` replay buffers produced by `scripts/run_expert_experience.py`. Loads experiences, segments them into episodes, and checks that the expert agent behaves as expected:

1. Every episode terminates with the `STOP` action (`STOP = code.n`, see `src/core/environment.py`).
2. No bit-flip action is repeated within an episode (omniscient / min-sum-expert plan a distinct set of flips).
3. Episode lengths stay within `environment.max_steps` from the experiment configuration.
4. Most episodes terminate with a positive final reward (the `ImprovedRewardFunction` success bonus dominates a clean STOP).

Reward shape comes from `src/core/reward.py::ImprovedRewardFunction`.

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'pyproject.toml').exists() and (_p / 'src').exists():
        sys.path.insert(0, str(_p))
        break

import pickle
from collections import namedtuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.core.ldpc_code import LDPC_Code
from src.utils.path_utils import resolve_project_path

# Mirror of src/training/replay_buffer.py::Experience so unpickling works
# without importing the (heavier) replay-buffer module.
Experience = namedtuple('Experience', ('state', 'action', 'reward', 'next_state', 'done'))

## Configuration

`CODE_ID` selects the LDPC code so `STOP = code.n` is correct without inspecting the pickle. `MAX_STEPS` should match the `environment.max_steps` used when the expert experiences were generated.

In [ ]:
CODE_ID = '64x128'
EXPERT_FILE = 'data/64x128_omniscient_500000.pkl'
MAX_STEPS = 12         # environment.max_steps used during generation
EXPORT_CSV = False     # set True to write a flat CSV alongside the pkl

code = LDPC_Code.from_default(code_id=CODE_ID)
STOP_ACTION = code.n  # bit flip actions are 0..n-1, STOP is n

expert_path = resolve_project_path(EXPERT_FILE)
if not expert_path.exists():
    raise FileNotFoundError(f'Expert experiences not found: {expert_path}')
print(f'Code     : n={code.n}, k={code.k}, m={code.m}  (STOP action = {STOP_ACTION})')
print(f'File     : {expert_path}  ({expert_path.stat().st_size / (1024**2):.1f} MB)')

## Load and transform

Each experience becomes one row in the DataFrame. The `episode` column counts up on every `done=True` boundary; `step` resets at the start of each new episode. This is the only transformation in the notebook — everything else operates on this single long-form DataFrame.

In [ ]:
def load_experiences(path: Path) -> pd.DataFrame:
    with path.open('rb') as f:
        buffer = pickle.load(f)
    experiences = buffer['experiences']

    actions = np.fromiter((e.action for e in experiences), dtype=np.int64, count=len(experiences))
    rewards = np.fromiter((e.reward for e in experiences), dtype=np.float64, count=len(experiences))
    dones = np.fromiter((bool(e.done) for e in experiences), dtype=bool, count=len(experiences))

    # episode index = number of done boundaries strictly before this step
    episode_id = np.cumsum(np.concatenate(([0], dones[:-1]))).astype(np.int64)

    # step index = offset within the current episode
    step = np.empty(len(experiences), dtype=np.int64)
    start = 0
    for i, end in enumerate(np.flatnonzero(dones)):
        step[start:end + 1] = np.arange(end - start + 1)
        start = end + 1
    if start < len(experiences):  # trailing partial episode (no final done)
        step[start:] = np.arange(len(experiences) - start)

    return pd.DataFrame({
        'experience_id': np.arange(len(experiences), dtype=np.int64),
        'episode': episode_id,
        'step': step,
        'action': actions,
        'reward': rewards,
        'done': dones,
    })


df = load_experiences(expert_path)
n_episodes = int(df['done'].sum())
print(f'Experiences : {len(df):,}')
print(f'Episodes    : {n_episodes:,}')
print(f'Avg length  : {len(df) / max(n_episodes, 1):.2f} steps/episode')
df.head(10)

## Episode-level summary

Collapse the long-form DataFrame to one row per episode.

In [ ]:
def episode_summary(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby('episode', sort=True)
    return pd.DataFrame({
        'length':        grouped.size(),
        'total_reward':  grouped['reward'].sum(),
        'final_reward':  grouped['reward'].last(),
        'final_action':  grouped['action'].last(),
        'bit_flips':     grouped['action'].apply(lambda s: int((s < STOP_ACTION).sum())),
        'unique_flips':  grouped['action'].apply(lambda s: int(s[s < STOP_ACTION].nunique())),
    })


episodes = episode_summary(df)
episodes.describe()

## Quality checks

Four checks aligned with the expert-agent contract.

In [ ]:
def quality_report(episodes: pd.DataFrame, max_steps: int) -> pd.DataFrame:
    total = len(episodes)
    end_with_stop = (episodes['final_action'] == STOP_ACTION).sum()
    no_repeats = (episodes['bit_flips'] == episodes['unique_flips']).sum()
    within_limit = (episodes['length'] <= max_steps).sum()
    positive_final = (episodes['final_reward'] > 0).sum()

    checks = [
        ('all episodes end with STOP',              end_with_stop,  total, end_with_stop == total),
        ('no repeated bit-flips within an episode', no_repeats,     total, no_repeats == total),
        (f'length <= environment.max_steps ({max_steps})', within_limit, total, within_limit == total),
        ('positive final reward (>= 50%)',          positive_final, total, positive_final >= 0.5 * total),
    ]
    return pd.DataFrame(
        [(name, f'{passed}/{total}', f'{passed / total:.1%}', 'PASS' if ok else 'FAIL')
         for name, passed, total, ok in checks],
        columns=['check', 'passed', 'rate', 'status'],
    )


report = quality_report(episodes, MAX_STEPS)
overall = 'PASS' if (report['status'] == 'PASS').all() else 'FAIL'
print(f'Overall: {overall}\n')
report

## Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].hist(episodes['length'], bins=range(1, MAX_STEPS + 2), edgecolor='black', alpha=0.75)
axes[0, 0].set(title='Episode length', xlabel='steps', ylabel='count')
axes[0, 0].axvline(episodes['length'].mean(), color='red', linestyle='--',
                   label=f"mean = {episodes['length'].mean():.2f}")
axes[0, 0].legend()

bit_flips = df.loc[df['action'] < STOP_ACTION, 'action']
axes[0, 1].hist(bit_flips, bins=min(50, code.n), edgecolor='black', alpha=0.75)
axes[0, 1].set(title='Bit-flip action distribution', xlabel='bit position', ylabel='count')

axes[1, 0].hist(episodes['final_reward'], bins=40, edgecolor='black', alpha=0.75)
axes[1, 0].axvline(0, color='red', linestyle='--')
axes[1, 0].set(title='Final reward', xlabel='reward', ylabel='count')

axes[1, 1].scatter(episodes['length'], episodes['final_reward'], alpha=0.3, s=10)
axes[1, 1].axhline(0, color='red', linestyle='--')
axes[1, 1].set(title='Length vs final reward', xlabel='length', ylabel='final reward')

fig.tight_layout()
plt.show()

In [ ]:
def plot_reward_trajectories(df: pd.DataFrame, episode_ids):
    fig, ax = plt.subplots(figsize=(11, 6))
    for episode_id in episode_ids:
        ep = df[df['episode'] == episode_id].sort_values('step')
        if ep.empty:
            continue
        ax.plot(ep['step'], ep['reward'], marker='o', label=f'episode {episode_id}')
    ax.axhline(0, color='red', linestyle='--', alpha=0.4)
    ax.set(title='Per-step reward', xlabel='step', ylabel='reward')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')
    fig.tight_layout()
    plt.show()


# Sample a handful of episodes across the file for a visual sanity check.
if n_episodes:
    sample_ids = np.linspace(0, n_episodes - 1, num=min(6, n_episodes), dtype=int)
    plot_reward_trajectories(df, sample_ids)

## Manual inspection

In [ ]:
def show_episode(df: pd.DataFrame, episode_id: int) -> pd.DataFrame:
    ep = df[df['episode'] == episode_id].copy()
    if ep.empty:
        raise IndexError(f'episode {episode_id} not present (max: {df["episode"].max()})')
    ep['action_kind'] = np.where(ep['action'] < STOP_ACTION,
                                 'flip_' + ep['action'].astype(str),
                                 'STOP')
    return ep[['step', 'action', 'action_kind', 'reward', 'done']].reset_index(drop=True)


show_episode(df, 0)

In [ ]:
# Episodes most worth inspecting: longest and any with non-positive final reward.
longest = episodes.nlargest(5, 'length')
print('Longest episodes:')
print(longest.to_string())

negative = episodes[episodes['final_reward'] <= 0]
print(f'\nEpisodes with non-positive final reward: {len(negative)}')
if not negative.empty:
    print(negative.head(10).to_string())

## Optional: export

Writes a flat CSV next to the original `.pkl` for use in external tools. Disabled by default to avoid side-effects on notebook run.

In [ ]:
if EXPORT_CSV:
    csv_path = expert_path.with_suffix('.csv')
    df.to_csv(csv_path, index=False)
    print(f'Wrote {csv_path}')
else:
    print('EXPORT_CSV is False — skipping CSV export.')